# The transformer encoder
1. One encoder block with the paper's sizes (d_model = 512, 8 heads, d_ff = 2048): shapes and parameter counts.
2. "how are you" traced through a tiny block (d_model = 4) by hand in NumPy, checked against the Keras block.
3. The feed-forward network works on each word separately, with the same weights.
4. Residual connections: deep stacks of encoder blocks with and without them, trained on IMDB reviews.

In [1]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import gc
import numpy as np
import pandas as pd
import keras
import tensorflow as tf

np.set_printoptions(precision=3, suppress=True)
tf.config.experimental.enable_tensor_float_32_execution(False)   # exact float32 on the GPU, for the by-hand checks


def positional_encoding(n_pos, d_model):
    """PE[pos, 2i] = sin(pos / 10000^(2i/d)), PE[pos, 2i+1] = cos(same angle) (Vaswani et al. 2017, 3.5)."""
    pos = np.arange(n_pos)[:, None]
    i = np.arange(d_model // 2)[None, :]
    angle = pos / 10000 ** (2 * i / d_model)
    pe = np.zeros((n_pos, d_model))
    pe[:, 0::2] = np.sin(angle)
    pe[:, 1::2] = np.cos(angle)
    return pe.astype("float32")


class EncoderBlock(keras.layers.Layer):
    """Multi-head attention -> add & norm -> feed-forward (ReLU) -> add & norm (Vaswani et al. 2017, 3.1).
    residual=False drops the two "x +" paths and keeps everything else."""
    def __init__(self, d_model, heads, d_ff, residual=True, **kw):
        super().__init__(**kw)
        self.residual = residual
        self.mha = keras.layers.MultiHeadAttention(num_heads=heads, key_dim=d_model // heads)
        self.norm1 = keras.layers.LayerNormalization()
        self.ffn1 = keras.layers.Dense(d_ff, activation="relu")
        self.ffn2 = keras.layers.Dense(d_model)
        self.norm2 = keras.layers.LayerNormalization()

    def call(self, x):
        z = self.mha(x, x)
        z_norm = self.norm1(x + z if self.residual else z)
        y = self.ffn2(self.ffn1(z_norm))
        return self.norm2(z_norm + y if self.residual else y)

I0000 00:00:1791078268.100450 1402657 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


## 1. One encoder block with the paper's sizes
Input: 3 words ("how are you"), each a 512-number vector. Every sub-layer keeps the shape (3, 512).

In [2]:
d, h, d_ff, n = 512, 8, 2048, 3
keras.utils.set_random_seed(0)
X = np.random.default_rng(0).normal(size=(1, n, d)).astype("float32")   # stand-in for embedding + position
block = EncoderBlock(d, h, d_ff)
Y = block(X)
z = block.mha(X, X)
z_norm = block.norm1(X + z)
hidden = block.ffn1(z_norm)
print("X", X.shape[1:], " Z", z.shape[1:], " Z_norm", z_norm.shape[1:], " hidden layer of the FFN", hidden.shape[1:],
      " output", Y.shape[1:])

X (3, 512)  Z (3, 512)  Z_norm (3, 512)  hidden layer of the FFN (3, 2048)  output (3, 512)


Parameter counts: by formula and by Keras.

In [3]:
formula = {"multi-head attention": 4 * d * d + 4 * d,
           "feed-forward network": d * d_ff + d_ff + d_ff * d + d,
           "two layer norms": 2 * 2 * d}
keras_count = {"multi-head attention": block.mha.count_params(),
               "feed-forward network": block.ffn1.count_params() + block.ffn2.count_params(),
               "two layer norms": block.norm1.count_params() + block.norm2.count_params()}
counts = pd.DataFrame({"formula": formula, "Keras": keras_count})
counts.loc["one block"] = counts.sum()
counts.loc["6 blocks"] = 6 * counts.loc["one block"]
print(counts)
assert (counts["formula"] == counts["Keras"]).all()
assert block.count_params() == counts.loc["one block", "Keras"]
print("share of the feed-forward network in one block:",
      round(formula["feed-forward network"] / counts.loc["one block", "formula"], 3))
counts.to_csv("data/param_counts.csv")

                       formula     Keras
multi-head attention   1050624   1050624
feed-forward network   2099712   2099712
two layer norms           2048      2048
one block              3152384   3152384
6 blocks              18914304  18914304
share of the feed-forward network in one block: 0.666


Six blocks: separate copies, separate weights. The output shape stays (3, 512).

In [4]:
blocks = [EncoderBlock(d, h, d_ff) for _ in range(6)]
x = X
for b in blocks:
    x = b(x)
print("after 6 blocks:", x.shape[1:], " parameters:", sum(b.count_params() for b in blocks))
print("first weight of W_1 in blocks 1 and 2:", float(blocks[0].ffn1.kernel[0, 0]), float(blocks[1].ffn1.kernel[0, 0]))

after 6 blocks: (3, 512)  parameters: 18914304
first weight of W_1 in blocks 1 and 2: 0.03761610388755798 -0.034443773329257965


## 2. "how are you" through a tiny block, by hand
d_model = 4, 2 heads of 2, d_ff = 8. The embeddings are made-up example values.

In [5]:
words = ["how", "are", "you"]
E = np.array([[0.5, 1.0, -0.5, 0.2],
              [-1.0, 0.3, 0.8, 0.4],
              [0.7, -0.6, 0.1, 0.9]], dtype="float32")
PE4 = positional_encoding(3, 4)
X4 = E + PE4
print("positional encodings:\n", PE4, "\nX = embedding + position:\n", X4)
keras.utils.set_random_seed(1)
tiny = EncoderBlock(4, 2, 8)
out_keras = tiny(X4[None])[0].numpy()


def softmax_rows(S):
    W = np.exp(S - S.max(-1, keepdims=True))
    return W / W.sum(-1, keepdims=True)


def layer_norm(v, gamma, beta, eps=1e-3):                 # Keras' default epsilon
    mu, var = v.mean(-1, keepdims=True), v.var(-1, keepdims=True)
    return gamma * (v - mu) / np.sqrt(var + eps) + beta


Wq, bq, Wk, bk, Wv, bv, Wo, bo = tiny.mha.get_weights()
heads = []
for i in range(2):
    Q, K, V = X4 @ Wq[:, i] + bq[i], X4 @ Wk[:, i] + bk[i], X4 @ Wv[:, i] + bv[i]
    heads.append(softmax_rows(Q @ K.T / np.sqrt(2)) @ V)
Z = np.concatenate(heads, axis=1) @ Wo.reshape(4, 4) + bo
Z_add = X4 + Z
Z_norm = layer_norm(Z_add, *tiny.norm1.get_weights())
W1, b1 = tiny.ffn1.get_weights()
W2, b2 = tiny.ffn2.get_weights()
H = np.maximum(0, Z_norm @ W1 + b1)
Yf = H @ W2 + b2
Y_add = Z_norm + Yf
Y_norm = layer_norm(Y_add, *tiny.norm2.get_weights())
for name, M in [("Z (attention)", Z), ("Z' = X + Z", Z_add), ("Z_norm", Z_norm), ("H (hidden, ReLU)", H),
                ("Y (feed-forward)", Yf), ("Y' = Z_norm + Y", Y_add), ("Y_norm (block output)", Y_norm)]:
    print(f"{name:22s} row 'how':", M[0].round(2))
print("mean and std of Z' row 'how':", Z_add[0].mean().round(3), Z_add[0].std().round(3))
print("largest difference, by hand vs Keras:", np.abs(Y_norm - out_keras).max())
assert np.allclose(Y_norm, out_keras, atol=1e-5)

positional encodings:
 [[ 0.     1.     0.     1.   ]
 [ 0.841  0.54   0.01   1.   ]
 [ 0.909 -0.416  0.02   1.   ]] 
X = embedding + position:
 [[ 0.5    2.    -0.5    1.2  ]
 [-0.159  0.84   0.81   1.4  ]
 [ 1.609 -1.016  0.12   1.9  ]]


Z (attention)          row 'how': [ 1.08  0.77  0.39 -1.21]
Z' = X + Z             row 'how': [ 1.58  2.77 -0.11 -0.01]
Z_norm                 row 'how': [ 0.44  1.43 -0.98 -0.89]
H (hidden, ReLU)       row 'how': [0.2  0.   0.83 0.09 0.21 1.36 0.   0.47]
Y (feed-forward)       row 'how': [-0.78  0.67 -0.14 -0.31]
Y' = Z_norm + Y        row 'how': [-0.34  2.1  -1.12 -1.2 ]
Y_norm (block output)  row 'how': [-0.15  1.68 -0.73 -0.79]
mean and std of Z' row 'how': 1.058 1.195
largest difference, by hand vs Keras: 4.628040922716181e-07


## 3. The feed-forward network works on each word separately
Row by row gives the same as the whole matrix, and changing one word leaves the other rows' FFN output unchanged.

In [6]:
ffn = lambda M: tiny.ffn2(tiny.ffn1(M)).numpy()
whole = ffn(Z_norm[None])[0]
rows = np.vstack([ffn(Z_norm[i:i + 1][None])[0] for i in range(3)])
changed = Z_norm.copy(); changed[2] += 5.0                 # change only "you"
after = ffn(changed[None])[0]
print("row by row = whole matrix:", np.allclose(whole, rows, atol=1e-6))
print("'how' and 'are' unchanged when 'you' changes:", np.allclose(whole[:2], after[:2], atol=1e-6),
      "  'you' changed by", np.abs(whole[2] - after[2]).max().round(2))
assert np.allclose(whole, rows, atol=1e-6) and np.allclose(whole[:2], after[:2], atol=1e-6)

row by row = whole matrix: True
'how' and 'are' unchanged when 'you' changes: True   'you' changed by 7.13


Attention is the only place where words exchange information: changing "you" changes every row of Z.

In [7]:
X_changed = X4.copy(); X_changed[2] += 5.0
Z_changed = tiny.mha(X_changed[None], X_changed[None])[0].numpy()
print("change in Z per word:", np.abs(Z_changed - Z).max(axis=1).round(3))

change in Z per word: [1.591 4.116 5.747]


## 4. Residual connections
### 4a. Word vectors through untrained stacks, with and without the residual paths
A real IMDB review (its first 30 words), random embeddings of 64 numbers, blocks with 4 heads and d_ff = 256.
Measured after each block: the mean cosine similarity between the words, and between each word and its own input.
5 random starts, averaged. (No positional encoding here, so that the input words start out unrelated.)

In [8]:
V = 10000
(xtr, ytr), (xte, yte) = keras.datasets.imdb.load_data(num_words=V)
index = {i + 3: w for w, i in keras.datasets.imdb.get_word_index().items()}
review = np.array(xtr[0][1:31])
print(" ".join(index.get(i, "?") for i in review))


def unit(M):
    return M / np.linalg.norm(M, axis=1, keepdims=True)


def between_words(M):
    C = unit(M) @ unit(M).T
    n = len(M)
    return (C.sum() - n) / (n * n - n)


rows = []
for residual in (True, False):
    for seed in range(5):
        keras.utils.set_random_seed(seed)
        x0 = keras.layers.Embedding(V, 64)(review[None])[0].numpy()
        x = x0[None]
        rows.append(dict(residual=residual, seed=seed, blocks=0, between=between_words(x0), own=1.0))
        for k in range(1, 25):
            x = EncoderBlock(64, 4, 256, residual)(x)
            xs = x[0].numpy()
            rows.append(dict(residual=residual, seed=seed, blocks=k, between=between_words(xs),
                             own=(unit(xs) * unit(x0)).sum(1).mean()))
collapse = pd.DataFrame(rows).groupby(["residual", "blocks"])[["between", "own"]].mean().reset_index()
collapse.to_csv("data/collapse.csv", index=False)
print(collapse[collapse.blocks.isin([0, 1, 2, 6, 12, 24])].round(3).to_string(index=False))

this film was just brilliant casting location scenery story direction everyone's really suited the part they played and you could just imagine being there robert ? is an amazing actor


 residual  blocks  between    own
    False       0   -0.002  1.000
    False       1    1.000 -0.008
    False       2    1.000  0.004
    False       6    1.000  0.013
    False      12    1.000 -0.000
    False      24    1.000  0.012
     True       0   -0.002  1.000
     True       1    0.089  0.857
     True       2    0.220  0.702
     True       6    0.833  0.207
     True      12    0.996  0.024
     True      24    1.000  0.003


### 4b and 4c. Six blocks trained on IMDB reviews
Reviews cut to their last 100 words (10,000-word vocabulary), embeddings of 32 numbers + positional encoding,
6 encoder blocks (2 heads, d_ff = 64), average over the words, one sigmoid output. Adam, learning rate 0.001,
batch 64, 3 epochs, 3 seeds; validation on the first 5,000 test reviews. The only difference: the residual paths.
First, the gradient that reaches the embedding layer before any training (one batch of 64 reviews).

In [9]:
L, D, H, DFF, N = 100, 32, 2, 64, 6
xtr_p = keras.utils.pad_sequences(xtr, maxlen=L)
xte_p = keras.utils.pad_sequences(xte, maxlen=L)
PE = positional_encoding(L, D)


def build(residual, seed):
    keras.utils.set_random_seed(seed)
    inp = keras.Input((L,), dtype="int32")
    emb = keras.layers.Embedding(V, D)
    x = emb(inp) + PE
    for _ in range(N):
        x = EncoderBlock(D, H, DFF, residual)(x)
    out = keras.layers.Dense(1, activation="sigmoid")(keras.layers.GlobalAveragePooling1D()(x))
    return keras.Model(inp, out), emb


grads = []
xb, yb = tf.constant(xtr_p[:64], dtype=tf.int32), tf.constant(ytr[:64], dtype=tf.float32)
for residual in (True, False):
    for seed in range(3):
        m, emb = build(residual, seed)
        with tf.GradientTape() as tape:
            loss = tf.reduce_mean(keras.losses.binary_crossentropy(yb, m(xb, training=True)[:, 0]))
        assert tuple(m.trainable_variables[0].shape) == (V, D)        # the embedding matrix comes first
        g = tape.gradient(loss, m.trainable_variables)[0]
        grads.append(dict(residual=residual, seed=seed, grad=float(tf.norm(tf.convert_to_tensor(g)))))
grads = pd.DataFrame(grads)
print(grads.groupby("residual").grad.agg(["mean", "min", "max"]))
grads.to_csv("data/embedding_grad.csv", index=False)

              mean       min       max
residual                              
False     0.066600  0.053149  0.083324
True      0.025273  0.012722  0.035110


In [10]:
curves = []
for residual in (True, False):
    for seed in range(3):
        keras.backend.clear_session(); gc.collect()
        m, _ = build(residual, seed)
        m.compile(optimizer=keras.optimizers.Adam(1e-3), loss="binary_crossentropy", metrics=["accuracy"])
        hist = m.fit(xtr_p, ytr, epochs=3, batch_size=64, validation_data=(xte_p[:5000], yte[:5000]), verbose=0).history
        for e in range(3):
            curves.append(dict(residual=residual, seed=seed, epoch=e + 1, train=hist["accuracy"][e], val=hist["val_accuracy"][e]))
        print(f"residual {residual!s:5s} seed {seed}: train accuracy {np.round(hist['accuracy'], 3)}, "
              f"validation {np.round(hist['val_accuracy'], 3)}")
curves = pd.DataFrame(curves)
curves.to_csv("data/training.csv", index=False)
print(curves.groupby(["residual", "epoch"])[["train", "val"]].mean().round(3))

residual True  seed 0: train accuracy [0.502 0.829 0.893], validation [0.623 0.849 0.825]


residual True  seed 1: train accuracy [0.547 0.855 0.903], validation [0.799 0.848 0.845]


residual True  seed 2: train accuracy [0.512 0.497 0.498], validation [0.486 0.486 0.486]


residual False seed 0: train accuracy [0.5   0.495 0.496], validation [0.514 0.514 0.514]


residual False seed 1: train accuracy [0.501 0.499 0.498], validation [0.514 0.486 0.486]


residual False seed 2: train accuracy [0.498 0.506 0.561], validation [0.486 0.486 0.486]
                train    val
residual epoch              
False    1      0.500  0.505
         2      0.500  0.495
         3      0.519  0.495
True     1      0.520  0.636
         2      0.727  0.728
         3      0.765  0.718
